In [4]:
import numpy as np

In [57]:
gen = CliqueGenerator(0.1,15,2,4)


In [7]:
bool_matrix = np.random.randint(0,2,(5,5))

In [8]:
bool_matrix

array([[1, 1, 1, 1, 1],
       [1, 1, 0, 1, 0],
       [0, 1, 0, 0, 0],
       [0, 1, 1, 0, 1],
       [0, 1, 1, 1, 1]])

In [9]:
bool_matrix.sum(axis=1)

array([5, 3, 1, 3, 4])

In [10]:
bool_matrix.sum(axis=0)

array([2, 5, 3, 3, 3])

In [58]:
adjacency_matrix, solutions, sizes = gen.create()

In [59]:
adjacency_matrix

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 1., 0., 0.

In [60]:
diag_cut = True
original = adjacency_matrix
if diag_cut:
    for i in range(adjacency_matrix.shape[0]):
        original[i][i]=0

In [61]:
original

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 1., 0., 0.

In [62]:
def reorder_adjacency_matrix(original, diag_cut=True):
    A = np.array(original, dtype=np.int32, copy=True)  # Kopie, kein bool-Problem mehr
    if diag_cut:
        np.fill_diagonal(A, 0)
    out_deg = A.sum(axis=1)
    post_order = np.argsort(out_deg, kind="stable")[::-1]
    amount_post_order = out_deg[post_order]  # garantiert konsistent zu post_order
    sorted_matrix = A[np.ix_(post_order, post_order)]
    original_order = np.arange(A.shape[0], dtype=np.int32)
    return sorted_matrix, original_order, post_order, amount_post_order

def get_mask_of_outgoing_connections(sorted_matrix, amount_post_order):
    k = int((amount_post_order >= 2).sum())
    sub = sorted_matrix[:k, :k]  # sortiert -> die ersten k Zeilen sind die nicht-trivialen
    out_matrix = sub * amount_post_order[:k, None]  # Zeile i * outdeg(i)
    return out_matrix, amount_post_order[:k], k

def get_mask_of_incoming_connections(sorted_matrix, k):
    sub = sorted_matrix[:k, :k]
    in_conns = sub.sum(axis=0)
    in_matrix = sub * in_conns[None, :]  # Spalte j * indeg(j), explizit
    return in_matrix, in_conns

def get_meta_information(adjacency_matrix):
    adjacency_matrix = adjacency_matrix
    sorted_matrix, original_order, post_order, amount_post_order = reorder_adjacency_matrix(adjacency_matrix)
    out_matrix, out_conns, last_non_trivial_row = get_mask_of_outgoing_connections(sorted_matrix, amount_post_order)
    in_matrix, in_conns = get_mask_of_incoming_connections(sorted_matrix,last_non_trivial_row)
    return sorted_matrix, original_order, post_order, amount_post_order,last_non_trivial_row, out_matrix, out_conns, in_matrix, in_conns


In [63]:
get_meta_information(adjacency_matrix)

(array([[0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0],
        [0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=int32),
 array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
       dtype=int32),
 array([ 2, 14,  6,  3,  7, 10,  1, 13, 12, 11,  5,  9,  8,  4,  0]),


In [51]:
class Solver():
    def __init__(self,
                 noise_level = 0.05, 
                 adjacency_matrix_size = 20, 
                 num_cliques = 5, 
                 max_clique_size =5,
                 adjacency_matrix=None,
                 info=False,
                 diag_cut=True,
                print_solutions=True):
        
        self.diag_cut = diag_cut
        if adjacency_matrix is None:
            self.gen = CliqueGenerator(noise_level = noise_level, 
                                   adjacency_matrix_size = adjacency_matrix_size, 
                                   num_cliques = num_cliques, 
                                   max_clique_size = max_clique_size,
                                   info=info)
            
        
            self.adjacency_matrix, self.solutions, self.clique_sizes = self.gen.create()
            print("generated!",self.adjacency_matrix)
            if print_solutions:
                print("Cliques at: ",self.solutions)
        else:
            self.adjacency_matrix = adjacency_matrix
        
        #from here on, only adjacency matrix is used. the code above is just to give you an example
        self.size=self.adjacency_matrix.shape[0]
        
        self.sorted_adjacency_matrix, self.original_order, self.post_order,self.amount_post_order,self.last_non_trivial_row, self.sorted_numerated_out_matrix, self.outgoing_connections, self.sorted_numerated_in_matrix, self.incoming_connections = get_meta_information(self.adjacency_matrix)
        
        self.current_order = self.post_order
        
        
        
        
        
        self.valid_index = np.arange(self.sorted_numerated_out_matrix.shape[0])
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        self.current_order = self.current_order[:len(self.outgoing_connections)]
        self.current_order = self.current_order[self.nodes_of_interest]
        self.sorted_numerated_out_matrix=self.sorted_numerated_out_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        
        
        self.sorted_numerated_in_matrix=self.sorted_numerated_in_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        
        
        
        
        self.valid_index=self.valid_index[self.nodes_of_interest]
        self.exclusion_prep = self.post_order[:len(self.outgoing_connections)]
        self.not_included = list(self.exclusion_prep[~self.nodes_of_interest])
        self.exclusion_prep = self.exclusion_prep[self.nodes_of_interest]
        self.outgoing_connections = self.outgoing_connections[self.nodes_of_interest]
        self.incoming_connections = self.incoming_connections[self.nodes_of_interest]
        self.nodes_of_interest = self.nodes_of_interest[self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        self.helper_adjacency_matrix = self.create_helper()
        self.cliques = []
        
        self.create_helper()
        self.added_for_removal = []
        self.update_in_out_conn()
        self.update_nodes_of_interest()
    
    
    def info(self):
        print("self.sorted_numerated_out_matrix",self.sorted_numerated_out_matrix.shape)
        print("self.sorted_numerated_in_matrix",self.sorted_numerated_in_matrix.shape)
        print("self.valid_index",self.valid_index.shape)
        print("self.exclusion_prep",self.exclusion_prep.shape)
        print("self.outgoing_connections",self.outgoing_connections.shape)
        print("self.incoming_connections",self.incoming_connections.shape)
        print("self.nodes_of_interest",self.nodes_of_interest.shape)
        print("self.valid_index",self.valid_index.shape)
        print("self.current_order",self.current_order.shape)
        print("self.post_order: ",self.post_order)
        print("self.current_order: ",self.current_order)
        print("self.outgoing_connections: ",self.outgoing_connections)
        print("self.incoming_connections: ",self.incoming_connections)
        print("-"*60,"\nself.sorted_numerated_out_matrix\n")
        self.out_matrix()
        print("-"*60,"\nself.sorted_numerated_in_matrix\n")
        self.in_matrix()
        print("-"*60)
        print("Exclusion Prep: ",self.exclusion_prep,"Not included:", self.not_included)
        
    def out_matrix(self):
        print(self.sorted_numerated_out_matrix,"\n",self.sorted_numerated_out_matrix.shape,"\n")
    def in_matrix(self):
        print(self.sorted_numerated_in_matrix,"\n",self.sorted_numerated_in_matrix.shape,"\n")

        
    # Bis hierher ok.
    def correct_out(self,col):
        mask = self.sorted_numerated_out_matrix[:,col]!=0
        self.sorted_numerated_out_matrix[:,col]-=mask
        self.outgoing_connections[col]-=1
        
    def correct_in(self,col):
        mask = self.sorted_numerated_in_matrix[:,col]!=0
        self.sorted_numerated_in_matrix[:,col]-=mask
        self.incoming_connections[col]-=1
    
    def reduce_single_in(self,col,row):
        self.sorted_numerated_in_matrix[col,row]-=1
    
    def reduce_single_out(self,col,row):
        self.sorted_numerated_out_matrix[col,row]-=1
        
    def remove_idx(self,idx):
        assert self.sorted_numerated_out_matrix.shape[0]>0
        idx_len = self.sorted_numerated_out_matrix.shape[0]
        valid_idx = np.ones(idx_len,dtype=np.int32)
        valid_idx[idx]=0
        valid_idx = np.bool(valid_idx)
        for i,b in enumerate(self.sorted_numerated_out_matrix[idx]!=0):
            if b:
                self.correct_in(i)
        self.sorted_numerated_out_matrix = self.sorted_numerated_out_matrix[valid_idx,:][:,valid_idx]
        self.sorted_numerated_in_matrix = self.sorted_numerated_in_matrix[valid_idx,:][:,valid_idx]
        
                
        self.not_included.append(list(self.exclusion_prep[~valid_idx]))
        self.exclusion_prep = self.exclusion_prep[valid_idx]
        self.outgoing_connections = self.outgoing_connections[valid_idx]
        self.incoming_connections = self.incoming_connections[valid_idx]
        self.current_order = self.current_order[valid_idx]
        self.nodes_of_interest = self.nodes_of_interest[valid_idx]
    
    def remove_job(self):#do this ONLY if everything else is done and you are tired of your collected data
        self.added_for_removal.sort()
        for i in self.added_for_removal:
            self.remove_idx(i)
        self.added_for_removal = []
        self.update_in_out_conn()
    
    def update_nodes_of_interest(self,delete=True):# same here, ackthuallhy it became the only and safe instance where you can do it
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        mask = ~self.nodes_of_interest
        for i,b in enumerate(mask):
            if b:
                if i not in self.added_for_removal:
                    self.added_for_removal.append(i)
        if delete:
            self.remove_job()
        
            
    def final_check_if_clique(self,real_indeces):#
        check = np.ones(len(real_indeces),dtype=np.int32)
        for count,i in enumerate(real_indeces):
            for j in real_indeces:
                if i!=j:
                    check[count] = (check[count]==self.adjacency_matrix[i][j])==1
        return check.sum()==len(real_indeces)
    
    def set_inf_clique(self,real_indices):#
        for i in real_indices:
            for j in real_indices:
                if i!=j:
                    self.adjacency_matrix[i][j]=np.inf
    
    def translator(self, candidates):#
        rgw = []
        for i in candidates:
            rgw.append(self.current_order[i].item())
        return np.array(rgw,dtype=np.int32)
    
    def pick_last(self):
        last_candidate = len(self.outgoing_connections)-1
        self.check_idx(last_candidate)            
    
    def create_helper(self):
        self.helper_adjacency_matrix = self.sorted_numerated_in_matrix != 0
        
    # Bis hierher absolut alles okay.    

    

    
    
    def update_in_out_conn(self):
        self.outgoing_connections = self.helper_adjacency_matrix.sum(axis=0)
        self.incoming_connections = self.helper_adjacency_matrix.sum(axis=1)
        
    def check_idx_part1(self,idx):#
        assert idx<len(self.outgoing_connections)# just to make sure we are in the right intervall
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
            
        
        lst = self.helper_adjacency_matrix[idx]
        
        sum_out = lst.sum()
        if sum_out <2:
            if idx not in self.added_for_removal:
                self.added_for_removal.append(idx)
            return idx, 0, [], {}
        check = 0           #here : counter for suitable candidates
        indices = []        #for the outer matrix
        
        for i,b in enumerate(lst):
            if b:
                if self.helper_adjacency_matrix[i][idx]:
                    indices.append(i)
                    check+=1
                else:
                    #self.sorted_numerated_out_matrix[i][idx]-=1 # legacy?
                    sum_out-=1
        tmp_bool_dict={}#to avoid new calculations
        if check<2:#this means it can never be a part of a clique, to reduce compute delete it.
            if idx not in self.added_for_removal:
                self.added_for_removal.append(idx)
            return idx, 0, [], {}

        for i in indices:
            tmp_bool_dict[i]=self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[i]
        return idx, check, indices, tmp_bool_dict
                    
                    
    def check_idx_part2(self,idx,check,indices,tmp_bool_dict):#
        check=check
        candidates = []
        for index in reversed(indices):
            tmp_bool_sum = tmp_bool_dict[index].sum()
            if tmp_bool_sum>0:
                tmp_bool_vec2 = tmp_bool_dict[index]
                tmp_bool_vec2[idx]=1
                tmp_bool_vec2[index]=1
                count = len(self.helper_adjacency_matrix[0])-1
            else:
                continue
                
            while tmp_bool_sum and count>=0:
                if tmp_bool_dict[index][count]==1 and count!=index and count!=idx:
                    tmp_bool_sum-=1
                    if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:#this may look like we iterate over the whole matrix, but we only take the ones
                        if count not in candidates and count!=idx and count!=index:
                            candidates.append(count)
                        else:
                            self.correct_in(idx)
                if index not in candidates:
                    candidates.append(index)
                count-=1
            check-=1
            if check<2 and len(candidates)<1:#clique begins with 3
                if idx not in self.added_for_removal:
                    self.added_for_removal.append(idx)
                return
        return candidates
    
    def check_idx_part3(self,idx,candidates):#
        candidates=candidates
        if idx not in candidates:
            candidates.append(idx)
        checkbox = np.zeros((len(candidates),len(candidates)))
        for i in range(len(candidates)-1):
            for j in range(len(candidates)-1):
                checkbox[i][j]=self.helper_adjacency_matrix[candidates[i]][candidates[j]]!=0
        checkbox[len(candidates)-1][len(candidates)-1]=1
        checkbox = np.array(checkbox,dtype=np.int32)
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        return checkbox, real_candidates
    
    
    
    
    # addieren der box + multiplizieren mit idx GEILER CHECK

    def check_idx_part4(self):
        pass
    
    


    def check_for_bigger_clique(self):
        # addiere echte matrix rows, nulle danach die einzelnen vorherigen bestandteile. wenn 
        # eine davon max erreicht, heißt das, dass die clique erweitert werden kann
        pass

    


    
    def check_idx(self,idx):
        print("idx:",idx)
        idx, check, indices, tmp_bool_dict = self.check_idx_part1(idx)
        candidates = self.check_idx_part2(idx, check, indices, tmp_bool_dict)
        checkbox, real_candidates = self.check_idx_part3(idx,candidates)
        real_indeces = self.translator(candidates)
        max_clique_size = self.check_how_big_a_clique_can_be(candidates)
        print(max_clique_size)
        
        """
        print("checkbox: ",checkbox)
        print("checkbox.sum(axis=1): ",checkbox.sum(axis=1))
        print("candidates: ",candidates)
        print("self.current_order[candidates]: ",self.current_order[candidates])
        print("self.adjacency_matrix[self.current_order[candidates]]: ",self.adjacency_matrix[self.current_order[candidates]])
        print("self.added_for_removal: ",self.added_for_removal)
        print("#################################")
        """
        
        #self.remove_job()
        #self.update_nodes_of_interest()
        
        
        

    def check_how_big_a_clique_can_be(self,candidates):
        check = np.zeros(len(candidates),dtype=np.int32)
        print("self.outgoing_connections",self.outgoing_connections)
        print("self.incoming_connections",self.incoming_connections)
        for i,candidate in enumerate(candidates):
            A = self.outgoing_connections[candidate]
            B = self.incoming_connections[candidate]
            check[i] = min(A,B)
        maximum = 2
        print(check)
        for i in range(len(check)+1,2,-1):
            if (check>=i).sum()>=i:
                if i>maximum:
                    maximum=i
                break
        
        return maximum
        

In [52]:
S = Solver()

generated! [[0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1.]
 [0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 1. 0. 1. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 

In [11]:
class CliqueGenerator():
    """
    Create an adjacency matrix with cliques in it to test our algorithm. It will have 'num_cliques' 
    cliques with random size between 3 and max_clique_size.
    This function returns the adjacency matrix, the solutions and the sizes of the cliques.
    """
    def __init__(self, 
                 noise_level, 
                 adjacency_matrix_size, 
                 num_cliques,
                 max_clique_size, 
                 info=False,
                diag_cut=False):
        
        self.noise_level = noise_level
        self.adjacency_matrix_size = adjacency_matrix_size
        self.num_cliques = num_cliques
        self.info = info
        self.max_clique_size = max_clique_size
        
    def create(self,num_cliques=None,diag_cut=True):
        if num_cliques is None:
            num_cliques = self.num_cliques
        assert num_cliques <= self.adjacency_matrix_size
        placeholder_array = np.zeros((self.adjacency_matrix_size,self.adjacency_matrix_size))
        clique_sizes = []
        for i in range(num_cliques):
            clique_sizes.append(int(np.random.randint(3,self.max_clique_size)))
        
        candidates = []
        i = 0
        while len(candidates)<num_cliques:
            candidate = []
            while len(candidate) < clique_sizes[i]:
                tmp = int(np.random.randint(0,self.adjacency_matrix_size))
                if tmp not in candidate:
                    candidate.append(tmp)
                candidate.sort()
            if candidate not in candidates:
                candidates.append(candidate)
                i+=1
        
        
        noise = int((1-self.noise_level)*100)
        noise_matrix = np.random.randint(0,100,size=(self.adjacency_matrix_size,self.adjacency_matrix_size))>noise
        difference = noise_matrix-placeholder_array*noise_matrix
        placeholder_array += difference
        for candidate in candidates:
            for i in candidate:
                for j in candidate:
                    if j!=i:
                        placeholder_array[i][j]=1
        if self.info:
            print("[",len(clique_sizes),"]","Cliques will be created in the following sizes: ",*clique_sizes,"\n\n","-"*30)
            print("The solutions will be: ", candidates,"\n\n","-"*30,"\n")  
            print("Now we have a pure adjacency matrix without any noise:","\n\n","-"*30,"\n",placeholder_array,"\n\n","-"*30)


        
        
        if self.info:
            print("\n\n","-"*30,"\nNow we have a pure adjacency matrix with noise:","\n\n","-"*30,"\n",placeholder_array,"\n\n","-"*30)
        if diag_cut:
            for i in range(placeholder_array.shape[0]):
                placeholder_array[i][i]=0#edges from i to i don't contribute to the solution, so we will ignore them
        
        return placeholder_array, candidates, clique_sizes